# Autonomous Agent Prediction — Playground Competition Demo (AIDE Agent) #

In this demo, you will learn how to evaluate an autonomous ML agent inspired by the [**AI-Driven Exploration (AIDE)**](https://github.com/wecoai/aideml) architecture. You could also explore the submissions to [MLE-bench](https://github.com/openai/mle-bench) for more ideas for agent designs.

### Overview ###

This example demonstrates the main components of an AIDE agent competition submission:

- **`submission/`**: The participant's agent definition.
  - **`agent.yaml`**: Root agent definition (`LoopAgent` driving iterative search cycles).
  - **`sub_agents/`**: Multi-agent team configuration (`tree_manager`, `coding_operator`, `evaluator`, `reviewer`).
  - **`configs/`**: LLM sampling & thinking configurations.
  - **`prompts/`**: Prompt templates (e.g. `coding_operator.md`).
  - **`skills/`**: AIDE Solution Tree Manager skill (`aide-manager`) containing python scripts for search policy, data preview, evaluation, and feedback update.

## Define the Agent Submission ##

Here, we create the directory structure and write the configuration files, system prompts, subagent definitions, and custom skill scripts that make up the autonomous AIDE ML agent. After the agent is defined, we create an archive `submission.zip` suitable for submission.

**You *do not* need to submit via a notebook.** You can also upload a `submission.zip` file directly. This notebook is meant to give you a starting point and illustrate the format.

In [1]:
import os

os.makedirs("submission/configs", exist_ok=True)
os.makedirs("submission/prompts", exist_ok=True)
os.makedirs("submission/sub_agents", exist_ok=True)
os.makedirs("submission/skills/aide-manager/scripts", exist_ok=True)
print("✅ Submission directory tree created: submission/")

✅ Submission directory tree created: submission/


In [2]:
%%writefile submission/agent.yaml
agent_class: LoopAgent
name: aide_root_agent
max_iterations: 25
sub_agents:
  - config_path: sub_agents/aide_cycle.yaml

Writing submission/agent.yaml


In [3]:
%%writefile submission/configs/sampling.yaml
temperature: 0.2
top_p: 0.95
top_k: 40
max_output_tokens: 8192

Writing submission/configs/sampling.yaml


In [4]:
%%writefile submission/configs/sampling_thinking.yaml
temperature: 0.5
top_p: 0.95
top_k: 40
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true

Writing submission/configs/sampling_thinking.yaml


In [5]:
%%writefile submission/sub_agents/aide_cycle.yaml
agent_class: SequentialAgent
name: aide_cycle
sub_agents:
  - config_path: sub_agents/tree_manager.yaml
  - config_path: sub_agents/coding_operator.yaml
  - config_path: sub_agents/evaluator.yaml
  - config_path: sub_agents/reviewer.yaml

Writing submission/sub_agents/aide_cycle.yaml


In [6]:
%%writefile submission/sub_agents/tree_manager.yaml
agent_class: LlmAgent
name: tree_manager
model: gemini-3.1-flash-lite
instruction: |
  You are the Tree Manager Agent for AIDE.

  IMPORTANT: Your ONLY action is to call `run_skill_script(skill_name="aide-manager", file_path="scripts/aide_search_policy.py")`.
  Do NOT call any other scripts or tools. Do NOT call `aide_update_feedback.py` or `aide_evaluator.py`.

  Execute `scripts/aide_search_policy.py`, capture its stdout, and output the resulting task prompt for the Coding Operator.
skills:
  - skills/aide-manager
generate_content_config: !include ../configs/sampling.yaml

Writing submission/sub_agents/tree_manager.yaml


In [7]:
%%writefile submission/sub_agents/coding_operator.yaml
agent_class: LlmAgent
name: coding_operator
model: gemini-3.5-flash
instruction: !include ../prompts/coding_operator.md
tools:
  - write_file
  - edit_file
  - read_file
generate_content_config: !include ../configs/sampling_thinking.yaml

Writing submission/sub_agents/coding_operator.yaml


In [8]:
%%writefile submission/prompts/coding_operator.md
You are the Coding Operator for an autonomous AI-Driven Exploration (AIDE) agent competing in a Kaggle machine learning competition.

## Competition Overview
{problem_description}
Target Metric: {metric_name} ({metric_direction})

## Sandbox Environment & Budget
You operate in an offline Linux container pre-installed with standard ML libraries. No internet access. `train.csv`, `test.csv`, and `sample_submission.csv` are in the working directory.
- Max Time: {max_time_minutes} minutes
- Token Budget: ${max_budget_usd} USD
- Max Submissions: {max_submissions}

## Available Packages
pandas, numpy, scikit-learn, xgboost, lightgbm, catboost, torch, tensorflow, scipy, statsmodels, category_encoders, optuna, feature_engine

## 🚨 MANDATORY REQUIREMENT
You MUST invoke `write_file` to save your generated Python code to `model_{NODE-ID}.py` BEFORE ending your turn.
- Do NOT write exploratory scripts like `eda.py`, test scripts, or helper files. ONLY write `model_{NODE-ID}.py`.
- If you do not call `write_file` for `model_{NODE-ID}.py`, your turn is wasted and marked as a failure.

## Instructions
You will receive a structured task prompt from the Tree Manager containing:
1. `TASK_TYPE`: DRAFT, DEBUG, or IMPROVE.
2. `NODE-ID`: The unique identifier integer for the new script to generate.
3. `BASE_SCRIPT`: The filepath of the parent script (if applicable).
4. `DATA_PREVIEW`: Detailed column information, data types, missing value counts, unique counts, and file sizes.
5. `SUMMARY`: Performance metrics, error tracebacks, or historical context.
6. `MEMORY`: Cross-iteration summary of all prior valid solution designs, empirical findings, and scores.

### Action Guidelines by Task Type:
- **If DRAFT**: You are a Kaggle grandmaster. Come up with an excellent, creative, and original approach that is DIFFERENT from all previous experiments listed in the MEMORY section. Start with a simple but thoughtful baseline solution. Do NOT call `read_file` on past model scripts — rely on `DATA_PREVIEW` and `MEMORY`. Use `write_file` to save `model_{NODE-ID}.py`.
- **If DEBUG**: Inspect the error traceback for `BASE_SCRIPT`. Identify the root bug (e.g. dtype mismatch, missing import, API change). Call `read_file` ONLY on `BASE_SCRIPT` if needed to see full source code. Fix the bug while preserving the core approach, and use `write_file` to save `model_{NODE-ID}.py`.
- **If IMPROVE**: Review `BASE_SCRIPT` and the MEMORY summary. Propose EXACTLY ONE atomic improvement (e.g. add a feature interaction, try frequency encoding, adjust regularization, or tune a hyperparameter). Call `read_file` ONLY on `BASE_SCRIPT`. Use `write_file` to save `model_{NODE-ID}.py`.

### Code Formatting Requirements:
1. Every generated Python script `model_{NODE-ID}.py` MUST begin with a **module-level docstring** (using `"""`) that concisely summarises your approach in 2-4 sentences.
2. Every script MUST include complete logic to train on `train.csv`, predict on `test.csv`, and save `submission_{NODE-ID}.csv`.
3. Ensure `submission_{NODE-ID}.csv` matches `sample_submission.csv` in format, column names, and row count.
4. Print `VALIDATION_SCORE: <score>` at the end of the script execution.
5. **Default Seed Zero**: Always use `0` for random seeds.
6. **Container Threading Limits**: Set `n_jobs=1` (or `thread_count=1`) in all ML estimators.
7. **Sandbox Pitfall Warnings**:
   - For XGBoost early stopping, use `early_stopping_rounds` parameter in `xgb.XGBClassifier(...)` / `xgb.XGBRegressor(...)` init or `xgb.callback.EarlyStopping()`.
   - Do NOT assign float values directly into `CategoricalDtype` columns — cast the column dtype first.
   - Always verify feature column names match between `train.csv` and `test.csv`.

Writing submission/prompts/coding_operator.md


In [9]:
%%writefile submission/sub_agents/evaluator.yaml
agent_class: LlmAgent
name: evaluator
model: gemini-3.1-flash-lite
instruction: |
  You are the Evaluator Agent for AIDE. Follow these steps in order:

  **Step 0 — Check budget.**
  Call `get_status()`. If `time_minutes_remaining` is less than 5 or `tool_calls_remaining` is less than 10:
  If any valid submissions exist, call `select_submission` with the best submission IDs and exit.

  **Step 1 — Run the evaluator script.**
  Call `run_skill_script(skill_name="aide-manager", file_path="scripts/aide_evaluator.py")`.
  Capture the full stdout.

  **CRITICAL GUARD**:
  If stdout contains `Evaluation failed: Script ... was not created by Coding Operator` or `No pending nodes to evaluate`:
  Do NOT retry, do NOT call `load_skill_resource`, and do NOT search for other scripts or tools.
  Simply output the status message and finish your turn.

  **Step 2 — Handle submission signals.**
  - If stdout contains `TO_SUBMIT: <filepath>`, call `submit_predictions(<filepath>)`.
    - If `submit_predictions` succeeds and returns a `submission_id` (e.g. `sub_1`), extract the NODE-ID from stdout (line `SUCCESS: Node <id> evaluated...`) and persist it:
      Call `run_skill_script(skill_name="aide-manager", file_path="scripts/aide_update_feedback.py", args=["--node-id", "<NODE-ID>", "--submission-id", "<submission_id>"])`.
  - If stdout contains `SELECT_SUBMISSIONS: <id1>, <id2>`, call `select_submission([<id1>, <id2>])`.

  Output a final summary of the evaluation and submission status.
tools:
  - submit_predictions
  - select_submission
  - get_status
skills:
  - skills/aide-manager
generate_content_config: !include ../configs/sampling.yaml

Writing submission/sub_agents/evaluator.yaml


In [10]:
%%writefile submission/sub_agents/reviewer.yaml
agent_class: LlmAgent
name: reviewer
model: gemini-3.1-flash-lite
instruction: |
  You are the Reviewer Agent for AIDE. You analyze execution results and produce structured assessments.

  Review the execution output from the previous evaluator step in the conversation history.

  **SKIP CHECK**:
  - If the evaluator output contains `No pending nodes to evaluate` or `Evaluation failed: Script ... was not created`, do NOT call `aide_update_feedback.py`. Output `SKIP: No new execution to review.` and finish.

  Identify the NODE-ID from the evaluator output (e.g. from `SUCCESS: Node <id> evaluated...` or `Execution failed for node <id>`).

  CRITICAL: You MUST call `run_skill_script` using `file_path="scripts/aide_update_feedback.py"`. Do NOT call `aide_evaluator.py`.

  Pass the following arguments array in `args`:
  `args: ["--node-id", "<NODE_ID>", "--is-bug", "<true|false>", "--analysis", "<2-3 sentence summary>", "--metric", "<float or null>", "--lower-is-better", "<true|false>"]`

  ## Assessment Criteria:
  - Set `is_bug = true` if ANY of:
    - The script raised an exception (returncode != 0)
    - The script failed to produce the submission CSV
    - The validation metric is NaN, negative, or clearly anomalous
    - The script ran but produced trivial/constant predictions
  - Set `is_bug = false` ONLY if the script trained a model, produced valid predictions, and reported a plausible validation score.

  After executing `aide_update_feedback.py`, output a brief summary of your review.
skills:
  - skills/aide-manager
generate_content_config: !include ../configs/sampling.yaml

Writing submission/sub_agents/reviewer.yaml


In [11]:
%%writefile submission/skills/aide-manager/SKILL.md
---
name: aide-manager
description: Manages the AIDE solution tree on disk, executes search policy π(T), generates cross-iteration memory summaries Σ(T), rich data previews, and evaluates/reviews candidate scripts.
---

# AIDE Manager Skill

This skill provides backend Python scripts to maintain the solution tree `solution_tree.json`, generate data previews, and evaluate candidate scripts inside the Docker sandbox.

## Available Scripts

### `scripts/aide_search_policy.py`
Reads `solution_tree.json`, applies the AIDE search policy heuristics (Draft → Debug → Improve), and outputs the formatted task prompt for the Coding Operator. Includes:
- Stochastic debug sampling (`DEBUG_PROB=0.5`) over leaf-only buggy nodes, creating child debug nodes (mirroring `aideml`'s `search_policy()`).
- `generate_memory_summary()` which serialises all valid nodes' plans, empirical findings, and scores as a `MEMORY:` block injected into every task prompt.
- Incorporates rich data preview via `aide_data_preview.py` in both DRAFT and DEBUG task prompts.
- Budget-aware search termination when remaining time or tool calls are low.
- Direction-aware best-node selection (`lower_is_better`).

### `scripts/aide_data_preview.py`
Generates a comprehensive dataset overview matching aideml's `data_preview.py`:
- File inventory with exact byte sizes.
- Detailed column schema for `train.csv` and `test.csv` (dtypes, null counts, unique counts, sample values).
- `sample_submission.csv` format preview.

### `scripts/aide_evaluator.py`
Executes the newly generated Python script, parses the validation score, stores the node's plan (from the script's module docstring), updates `solution_tree.json`, verifies `submission_{node_id}.csv` generation, and emits signals for the Evaluator agent (`TO_SUBMIT`, `SELECT_SUBMISSIONS`, `RAW_STDOUT_FOR_ANALYSIS`).

### `scripts/aide_update_feedback.py`
Persists structured LLM feedback (from the Reviewer agent) back into `solution_tree.json`:
- `--is-bug`: LLM verdict on bugginess.
- `--analysis`: 2-3 sentence qualitative empirical summary.
- `--metric`: Validation score float or null.
- `--lower-is-better`: Metric optimization direction.
- `--submission-id`: Competition harness-assigned submission ID (e.g. `sub_1`).

**Usage via `run_skill_script`**:
```python
run_skill_script(
    skill_name="aide-manager",
    file_path="scripts/aide_update_feedback.py",
    args=[
        "--node-id", "<id>",
        "--is-bug", "<true|false>",
        "--analysis", "<summary>",
        "--metric", "<score>",
        "--lower-is-better", "<true|false>",
        "--submission-id", "<sub_id>",
    ],
)
```

Writing submission/skills/aide-manager/SKILL.md


In [12]:
%%writefile submission/skills/aide-manager/scripts/aide_data_preview.py
"""Rich data preview generator matching aideml's data_preview.py."""
import os
import json
import pandas as pd
import numpy as np

WORK_DIR = "/work"
MAX_PREVIEW_CHARS = 6000

def generate_preview():
    parts = []

    # 1. File listing
    parts.append("=== FILES ===")
    if os.path.exists(WORK_DIR):
        for f in sorted(os.listdir(WORK_DIR)):
            fpath = os.path.join(WORK_DIR, f)
            if os.path.isfile(fpath):
                size = os.path.getsize(fpath)
                parts.append(f"  {f} ({size:,} bytes)")
    else:
        parts.append("  /work directory not found")

    # 2. CSV column analysis
    for csv_name in ["train.csv", "test.csv"]:
        csv_path = os.path.join(WORK_DIR, csv_name)
        if not os.path.exists(csv_path):
            continue
        try:
            df = pd.read_csv(csv_path, nrows=1000)
            parts.append(f"\n=== {csv_name} ({df.shape[0]}+ rows, {df.shape[1]} cols) ===")
            for col in df.columns:
                dtype = str(df[col].dtype)
                nulls = int(df[col].isnull().sum())
                nuniq = int(df[col].nunique())
                samples = df[col].dropna().head(3).tolist()
                parts.append(f"  {col}: {dtype}, {nulls} nulls, {nuniq} unique, samples={samples}")
        except Exception as e:
            parts.append(f"\n=== {csv_name} (error reading: {e}) ===")

    # 3. Sample submission format
    sub_path = os.path.join(WORK_DIR, "sample_submission.csv")
    if os.path.exists(sub_path):
        try:
            sub = pd.read_csv(sub_path, nrows=3)
            parts.append(f"\n=== sample_submission.csv ===")
            parts.append(f"  Columns: {list(sub.columns)}")
            parts.append(f"  {sub.head(3).to_string(index=False)}")
        except Exception as e:
            parts.append(f"\n=== sample_submission.csv (error reading: {e}) ===")

    preview = "\n".join(parts)
    if len(preview) > MAX_PREVIEW_CHARS:
        preview = preview[:MAX_PREVIEW_CHARS] + "\n... (truncated)"
    return preview

if __name__ == "__main__":
    print(generate_preview())

Writing submission/skills/aide-manager/scripts/aide_data_preview.py


In [13]:
%%writefile submission/skills/aide-manager/scripts/aide_evaluator.py
import os
import json
import subprocess
import sys

WORK_DIR = "/work"
TREE_FILE = os.path.join(WORK_DIR, "solution_tree.json")

def load_tree():
    if not os.path.exists(TREE_FILE):
        return {"nodes": {}, "next_id": 1, "draft_count": 0, "meta": {}}
    with open(TREE_FILE, "r") as f:
        tree = json.load(f)
        if "meta" not in tree:
            tree["meta"] = {}
        return tree

def save_tree(tree):
    with open(TREE_FILE, "w") as f:
        json.dump(tree, f, indent=2)

def parse_docstring_plan(filepath):
    """
    Extract the module-level docstring from a generated script as a plan summary.
    Mirrors aideml's node.plan.
    """
    try:
        with open(filepath, "r") as f:
            content = f.read()
        for delim in ('"""', "'''"):
            if content.lstrip().startswith(delim):
                parts = content.lstrip()[len(delim):].split(delim, 1)
                if parts:
                    return parts[0].strip()
    except Exception:
        pass
    return "No docstring plan provided."

def main():
    tree = load_tree()
    pending = [n for n in tree["nodes"].values() if n.get("status") == "pending"]
    if not pending:
        print("No pending nodes to evaluate.")
        return

    node = pending[-1]
    node_id = node["id"]
    script_file = os.path.join(WORK_DIR, f"model_{node_id}.py")
    sub_file = os.path.join(WORK_DIR, f"submission_{node_id}.csv")

    if not os.path.exists(script_file):
        node["status"] = "buggy"
        node["error"] = f"Script {script_file} was not created by Coding Operator."
        save_tree(tree)
        print(f"Evaluation failed: {node['error']}")
        return

    # Extract and store the plan from the script docstring before execution
    node["plan"] = parse_docstring_plan(script_file)
    save_tree(tree)

    print(f"Executing {script_file}...")
    result = subprocess.run(["python", script_file], capture_output=True, text=True, cwd=WORK_DIR)

    print("=== EXECUTION STDOUT ===")
    print(result.stdout[:4000])
    print("=== EXECUTION STDERR ===")
    print(result.stderr[:2000])

    if result.returncode != 0:
        node["status"] = "buggy"
        node["error"] = result.stderr[:2000]
        save_tree(tree)
        print(f"Execution failed for node {node_id}:\n{node['error']}")
        return

    if not os.path.exists(sub_file):
        node["status"] = "buggy"
        node["error"] = f"Script executed successfully but failed to generate {sub_file}."
        save_tree(tree)
        print(f"Evaluation failed: {node['error']}")
        return

    score = 0.0
    for line in result.stdout.split("\n"):
        if "VALIDATION_SCORE:" in line:
            try:
                score = float(line.split(":")[1].strip())
            except Exception:
                pass

    node["status"] = "valid"
    node["score"] = score
    node["submission_file"] = sub_file
    save_tree(tree)

    print(f"SUCCESS: Node {node_id} evaluated valid with internal score {score}.")
    print(f"TO_SUBMIT: {sub_file}")

    print("=== RAW_STDOUT_FOR_ANALYSIS ===")
    print(result.stdout[:4000])

    # Direction-aware submission selection
    valid_nodes = [n for n in tree["nodes"].values() if n.get("status") == "valid"]
    if len(valid_nodes) >= 2:
        lower_is_better = tree.get("meta", {}).get("lower_is_better", False)
        if lower_is_better:
            valid_nodes.sort(key=lambda x: x.get("score", float('inf')), reverse=False)
        else:
            valid_nodes.sort(key=lambda x: x.get("score", -float('inf')), reverse=True)

        # Prefer actual harness-assigned submission_id if available, fallback to sub_{id}
        top_ids = [n.get("submission_id", f"sub_{n['id']}") for n in valid_nodes[:2]]
        print(f"SELECT_SUBMISSIONS: {', '.join(top_ids)}")

if __name__ == "__main__":
    main()

Writing submission/skills/aide-manager/scripts/aide_evaluator.py


In [14]:
%%writefile submission/skills/aide-manager/scripts/aide_search_policy.py
import os
import json
import random
import sys

# Import rich data preview helper from same directory
script_dir = os.path.dirname(os.path.abspath(__file__))
if script_dir not in sys.path:
    sys.path.insert(0, script_dir)
from aide_data_preview import generate_preview

WORK_DIR = "/work"
TREE_FILE = os.path.join(WORK_DIR, "solution_tree.json")
MAX_DRAFTS = 5
MAX_DEBUG_DEPTH = 3
DEBUG_PROB = 0.5

def load_tree():
    if not os.path.exists(TREE_FILE):
        return {"nodes": {}, "next_id": 1, "draft_count": 0, "meta": {}}
    with open(TREE_FILE, "r") as f:
        tree = json.load(f)
        if "meta" not in tree:
            tree["meta"] = {}
        return tree

def save_tree(tree):
    with open(TREE_FILE, "w") as f:
        json.dump(tree, f, indent=2)

def generate_memory_summary(nodes):
    """
    Mirrors aideml's Journal.generate_summary().
    Iterates over all valid nodes and emits their plan, empirical findings, and score.
    Returns an empty string when no valid nodes exist yet.
    """
    valid_nodes = [n for n in nodes.values() if n.get("status") == "valid"]
    if not valid_nodes:
        return "No completed experiments yet."

    parts = []
    for n in sorted(valid_nodes, key=lambda x: x["id"]):
        part = f"Node {n['id']} ({n['type']})"
        if n.get("plan"):
            part += f"\n  Design: {n['plan']}"
        if n.get("analysis"):
            part += f"\n  Findings: {n['analysis']}"
        part += f"\n  Validation Score: {n.get('score', 'N/A')}"
        parts.append(part)

    return "\n-------------------------------\n".join(parts)

def compute_debug_depth(nodes, parent_node):
    depth = 0
    cursor = parent_node
    while cursor and cursor.get("type") == "debug":
        depth += 1
        pid = cursor.get("parent")
        cursor = nodes.get(str(pid)) if pid is not None else None
    return depth

def main():
    tree = load_tree()
    nodes = tree["nodes"]
    meta = tree.get("meta", {})

    # Budget check
    budget = meta.get("budget", {})
    if budget.get("time_minutes_remaining", 999) < 5 or budget.get("tool_calls_remaining", 999) < 10:
        print("BUDGET_LOW: Stopping search due to remaining budget limits. Finalizing submissions.")
        return

    # Check if there's already a pending node that hasn't been evaluated
    pending = [n for n in nodes.values() if n.get("status") == "pending"]
    if pending:
        target = pending[-1]
        node_id = target["id"]
        memory_str = generate_memory_summary(nodes)
        print(f"TASK_TYPE: {target['type'].upper()}\nNODE-ID: {node_id}\nBASE_SCRIPT: model_{target.get('parent', 'None')}.py\nSUMMARY: Retrying pending task {target['type']}.\nMEMORY:\n{memory_str}")
        return

    memory_str = generate_memory_summary(nodes)
    preview = generate_preview()

    # Rule 1: Draft
    if tree.get("draft_count", 0) < MAX_DRAFTS:
        node_id = tree["next_id"]
        tree["next_id"] += 1
        tree["draft_count"] = tree.get("draft_count", 0) + 1
        tree["nodes"][str(node_id)] = {"id": node_id, "parent": None, "status": "pending", "type": "draft", "debug_depth": 0}
        save_tree(tree)

        print(f"TASK_TYPE: DRAFT\nNODE-ID: {node_id}\nBASE_SCRIPT: None\nDATA_PREVIEW:\n{preview}\nSUMMARY: Draft initial baseline solution # {tree['draft_count']}.\nMEMORY:\n{memory_str}")
        return

    # Rule 2: Stochastic Debug — only select buggy nodes whose script file actually exists on disk
    parent_ids = {n.get("parent") for n in nodes.values() if n.get("parent") is not None}
    buggy_leaves = [
        n for n in nodes.values()
        if n.get("status") == "buggy"
        and n["id"] not in parent_ids
        and compute_debug_depth(nodes, n) < MAX_DEBUG_DEPTH
        and os.path.exists(os.path.join(WORK_DIR, f"model_{n['id']}.py"))
    ]

    if buggy_leaves and random.random() < DEBUG_PROB:
        parent = random.choice(buggy_leaves)
        new_id = tree["next_id"]
        tree["next_id"] += 1
        debug_depth = compute_debug_depth(nodes, parent) + 1
        tree["nodes"][str(new_id)] = {
            "id": new_id,
            "parent": parent["id"],
            "status": "pending",
            "type": "debug",
            "debug_depth": debug_depth
        }
        save_tree(tree)
        print(f"TASK_TYPE: DEBUG\nNODE-ID: {new_id}\nBASE_SCRIPT: model_{parent['id']}.py\nDATA_PREVIEW:\n{preview}\nSUMMARY: Script model_{parent['id']}.py failed with error:\n{parent.get('error', 'Unknown error')}\nFix the bug while preserving the approach.\nMEMORY:\n{memory_str}")
        return

    # Rule 3: Improve
    valid_nodes = [n for n in nodes.values() if n.get("status") == "valid"]
    if not valid_nodes:
        # Fallback if no valid nodes exist and max drafts reached
        node_id = tree["next_id"]
        tree["next_id"] += 1
        tree["nodes"][str(node_id)] = {"id": node_id, "parent": None, "status": "pending", "type": "draft", "debug_depth": 0}
        save_tree(tree)
        print(f"TASK_TYPE: DRAFT\nNODE-ID: {node_id}\nBASE_SCRIPT: None\nDATA_PREVIEW:\n{preview}\nSUMMARY: No valid nodes found. Draft a new baseline.\nMEMORY:\n{memory_str}")
        return

    lower_is_better = meta.get("lower_is_better", False)
    if lower_is_better:
        best_node = min(valid_nodes, key=lambda x: x.get("score", float('inf')))
    else:
        best_node = max(valid_nodes, key=lambda x: x.get("score", -float('inf')))

    new_id = tree["next_id"]
    tree["next_id"] += 1
    tree["nodes"][str(new_id)] = {"id": new_id, "parent": best_node["id"], "status": "pending", "type": "improve", "debug_depth": 0}
    save_tree(tree)

    print(f"TASK_TYPE: IMPROVE\nNODE-ID: {new_id}\nBASE_SCRIPT: model_{best_node['id']}.py\nSUMMARY: Best valid script model_{best_node['id']}.py achieved score {best_node.get('score')}. Propose exactly ONE atomic improvement (e.g. feature engineering, regularization, hyperparameter tuning).\nMEMORY:\n{memory_str}")

if __name__ == "__main__":
    main()

Writing submission/skills/aide-manager/scripts/aide_search_policy.py


In [15]:
%%writefile submission/skills/aide-manager/scripts/aide_update_feedback.py
"""
aide_update_feedback.py — Persist structured LLM feedback into solution_tree.json.

Mirrors the role of aideml's `node.analysis = response["summary"]` populated by
`review_func_spec`. Called by the Reviewer agent (and Evaluator) to store
qualitative analysis, bugginess verdict, validation metric, and metric direction.
"""
import os
import json
import argparse

WORK_DIR = "/work"
TREE_FILE = os.path.join(WORK_DIR, "solution_tree.json")

def load_tree():
    if not os.path.exists(TREE_FILE):
        return {"nodes": {}, "next_id": 1, "draft_count": 0, "meta": {}}
    with open(TREE_FILE, "r") as f:
        tree = json.load(f)
        if "meta" not in tree:
            tree["meta"] = {}
        return tree

def save_tree(tree):
    with open(TREE_FILE, "w") as f:
        json.dump(tree, f, indent=2)

def main():
    parser = argparse.ArgumentParser(
        description="Persist qualitative and structured feedback into solution_tree.json."
    )
    parser.add_argument(
        "--node-id",
        type=str,
        required=True,
        help="The node ID to attach the analysis to.",
    )
    parser.add_argument(
        "--analysis",
        type=str,
        default="",
        help="2-3 sentence qualitative summary of empirical findings.",
    )
    parser.add_argument(
        "--is-bug",
        type=str,
        default=None,
        choices=["true", "false", "True", "False"],
        help="Bugginess verdict from reviewer.",
    )
    parser.add_argument(
        "--metric",
        type=str,
        default=None,
        help="Parsed validation score (float or 'null').",
    )
    parser.add_argument(
        "--lower-is-better",
        type=str,
        default=None,
        choices=["true", "false", "True", "False"],
        help="Whether lower metric values are better.",
    )
    parser.add_argument(
        "--submission-id",
        type=str,
        default=None,
        help="The actual submission ID assigned by the competition harness (e.g. 'sub_1').",
    )
    args = parser.parse_args()

    tree = load_tree()
    node_key = str(args.node_id)

    if node_key not in tree["nodes"]:
        print(f"ERROR: Node {node_key} not found in solution_tree.json.")
        return

    target_node = tree["nodes"][node_key]

    if args.analysis:
        target_node["analysis"] = args.analysis

    if args.submission_id:
        target_node["submission_id"] = args.submission_id

    if args.metric and args.metric.lower() != "null":
        try:
            target_node["score"] = float(args.metric)
        except ValueError:
            pass

    if args.lower_is_better is not None:
        is_lower = args.lower_is_better.lower() == "true"
        tree["meta"]["lower_is_better"] = is_lower

    if args.is_bug is not None:
        is_bug = args.is_bug.lower() == "true"
        if is_bug:
            target_node["status"] = "buggy"
            if not target_node.get("error"):
                target_node["error"] = "Reviewer flagged node as buggy based on execution analysis."

    save_tree(tree)

    print(f"SUCCESS: Updated feedback for node {node_key}.")
    print(f"  Status       : {target_node['status']}")
    print(f"  Score        : {target_node.get('score', 'N/A')}")
    print(f"  Submission ID: {target_node.get('submission_id', 'N/A')}")
    print(f"  Lower Better : {tree['meta'].get('lower_is_better', False)}")
    print(f"  Analysis     : {target_node.get('analysis', '')}")

if __name__ == "__main__":
    main()

Writing submission/skills/aide-manager/scripts/aide_update_feedback.py


In [16]:
# Create the submission file
import subprocess

subprocess.run("cd submission && zip -r ../submission.zip . && cd -", shell=True, check=True)

  adding: prompts/ (stored 0%)
  adding: prompts/coding_operator.md (deflated 49%)
  adding: skills/ (stored 0%)
  adding: skills/aide-manager/ (stored 0%)
  adding: skills/aide-manager/SKILL.md (deflated 51%)
  adding: skills/aide-manager/scripts/ (stored 0%)
  adding: skills/aide-manager/scripts/aide_evaluator.py (deflated 63%)
  adding: skills/aide-manager/scripts/aide_search_policy.py (deflated 66%)
  adding: skills/aide-manager/scripts/aide_data_preview.py (deflated 62%)
  adding: skills/aide-manager/scripts/aide_update_feedback.py (deflated 64%)
  adding: configs/ (stored 0%)
  adding: configs/sampling_thinking.yaml (deflated 19%)
  adding: configs/sampling.yaml (deflated 6%)
  adding: sub_agents/ (stored 0%)
  adding: sub_agents/evaluator.yaml (deflated 50%)
  adding: sub_agents/aide_cycle.yaml (deflated 51%)
  adding: sub_agents/reviewer.yaml (deflated 48%)
  adding: sub_agents/coding_operator.yaml (deflated 34%)
  adding: sub_agents/tree_manager.yaml (deflated 38%)
  adding: a

CompletedProcess(args='cd submission && zip -r ../submission.zip . && cd -', returncode=0)